# W3D4 — The Same Network in PyTorch — Guided

**Week 3 · Day 4 · Deep Learning & Neural Networks** · Lab

Same data, same architecture, same starting weights, one tenth of the code.

Today's job is not to learn a library. It is to make the equivalence **concrete** — to load
yesterday's exact weights into a PyTorch model, train it with five lines, and watch the two loss
sequences agree to fourteen decimal places. Not "close". Identical.

Then to count what you deleted, and find that every deleted line came from one place: `backward()`.

Four things, in order:

- **`loss.backward()` prints `−0.1360` and `−0.0544`** — the two numbers you computed by nudging on
  Wednesday, from a library that was never told how.
- **`nn.Module`, `Dataset`, `DataLoader`** — the three objects that replace your loose functions and
  free-floating weight arrays.
- **The five-line loop**, run twice: once full-batch, matching yesterday exactly, and once with a
  `DataLoader` handing you 32 rows at a time.
- **The bug on purpose.** Delete `optimizer.zero_grad()`, retrain, and watch the curve do the thing
  slide 54 described. Then put it back.

You leave with `torch_net_history.parquet`.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٣ اليوم ٤ — الشبكة نفسها بـ PyTorch

**الأسبوع ٣ · اليوم ٤ · التعلّم العميق والشبكات العصبية** · معمل عملي

البيانات نفسها والمعمارية نفسها والأوزان الابتدائية نفسها، وعُشر الشيفرة.

ومهمّة اليوم ليست تعلّم مكتبة، بل جعل التكافؤ **ملموسًا**: أن تُحمّل أوزان الأمس بالضبط في نموذج
PyTorch، وتدرّبه بخمسة أسطر، وترى متتاليتَي الخسارة تتوافقان إلى أربع عشرة خانة عشرية. لا «قريبتان»
بل مطابقتان.

ثم أن تعدّ ما حذفته، فتجد أن كل سطر محذوف جاء من موضع واحد: `backward()`.

وأربعة أمور بالترتيب:

- **`loss.backward()` يطبع `−0.1360` و`−0.0544`** — الرقمان اللذان حسبتهما بالتحريك يوم الأربعاء، من
  مكتبة لم يُخبرها أحد كيف.
- **`nn.Module` و`Dataset` و`DataLoader`** — الكائنات الثلاثة التي تحلّ محلّ دوالك المتفرّقة
  ومصفوفات أوزانك السائبة.
- **حلقة الأسطر الخمسة** مشغّلة مرتين: مرة بدفعة كاملة تطابق الأمس تمامًا، ومرة بـ `DataLoader`
  يناولك ٣٢ صفًا في المرة.
- **العيب عن قصد.** احذف `optimizer.zero_grad()` وأعِد التدريب وشاهد المنحنى يفعل ما وصفته الشريحة
  ٥٤. ثم أعِده.

ستخرج بملف `torch_net_history.parquet`.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Create tensors, read their `.shape` and `.dtype`, and move them to a device.
- Use `requires_grad` and `.backward()`, and explain what `.grad` contains afterwards.
- Define a network as an `nn.Module` and enumerate its parameters.
- Load specific weights into a model, and say why `nn.Linear`'s weight matrix is transposed relative
  to yours.
- Write the five-line training loop from memory, in the right order.
- Say what `optimizer.zero_grad()` prevents, having removed it and looked at the damage.
- Wrap data in a `TensorDataset` and iterate it with a `DataLoader`, and explain what shuffling and
  batching change about the gradient.
- State honestly which lines a library removed and which it merely renamed.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- إنشاء مُوتِّرات (tensors) وقراءة `.shape` و`.dtype` ونقلها إلى جهاز.
- استخدام `requires_grad` و`.backward()`، وشرح ما يحتويه `.grad` بعدها.
- تعريف شبكة كـ `nn.Module` وتعداد معاملاتها.
- تحميل أوزان محدّدة في نموذج، وبيان لماذا تكون مصفوفة أوزان `nn.Linear` منقولةً مقارنةً بمصفوفتك.
- كتابة حلقة التدريب ذات الأسطر الخمسة من الذاكرة وبالترتيب الصحيح.
- بيان ما يمنعه `optimizer.zero_grad()` بعد أن حذفته ونظرت في الضرر.
- لفّ البيانات في `TensorDataset` والمرور عليها بـ `DataLoader`، وشرح ما يغيّره الخلط والتقسيم إلى
  دفعات في الاشتقاق.
- قول أيّ الأسطر حذفتها المكتبة وأيّها أعادت تسميتها فقط، قولًا صادقًا.

</div>


## About the data

**Dataset:** `moons_toy` — 500 rows × 3 columns, the same file as Tuesday and Wednesday, CC0

Nothing about the data changes today, and that is the point: the only way to compare two
implementations honestly is to change **one** thing, and today the one thing is the library.

You also load two artefacts from yesterday:

- **`numpy_net_params.npz`** — the weights your NumPy run **started** from. These go into the PyTorch
  model directly. Without them the two runs would start in different places and any agreement between
  them would be a coincidence.
- **`numpy_net_history.parquet`** — yesterday's 2,000 recorded losses, to plot against today's.

If you did not finish yesterday, `load_artefact` falls back to the reference copies in
`shared/solutions_cache/` and prints a note. You are not blocked.

**Watch out:** PyTorch defaults to `float32` and NumPy to `float64`. Mixing them raises an error that
names neither array, and — worse — a silent `float32` run will not match yesterday's `float64` numbers
past the sixth decimal. Today's model is explicitly `.double()` so the comparison is exact. In every
other week, `float32` is what you want.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `moons_toy` — ٥٠٠ صف × ٣ أعمدة، الملف نفسه يوم الثلاثاء والأربعاء، رخصة CC0

لا شيء في البيانات يتغيّر اليوم، وهذا هو المقصود: فالطريق الوحيد لمقارنة تنفيذين مقارنةً صادقة هو
تغيير شيء **واحد**، والشيء الواحد اليوم هو المكتبة.

وتُحمّل أيضًا أثرين من الأمس:

- **`numpy_net_params.npz`** وفيه الأوزان التي **بدأ** منها تشغيلك بـ NumPy. وتدخل هذه في نموذج
  PyTorch مباشرة. ولولاها لبدأ التشغيلان من موضعين مختلفين ولكان أي توافق بينهما مصادفة.
- **`numpy_net_history.parquet`** وفيه خسائر الأمس الألفان لترسمها مقابل خسائر اليوم.

وإن لم تُكمل الأمس فإن `load_artefact` ترجع إلى النسخ المرجعية في `shared/solutions_cache/` وتطبع
ملاحظة بذلك، فأنت غير متعطّل.

**انتبه:** يفترض PyTorch النوع `float32` وNumPy النوع `float64`. وخلطهما يُخرِج خطأً لا يُسمّي أيًّا
من المصفوفتين، والأسوأ أن تشغيلًا صامتًا بـ `float32` لن يطابق أرقام الأمس بـ `float64` بعد الخانة
السادسة. ونموذج اليوم `.double()` صراحةً لتكون المقارنة تامة. وفي كل أسبوع آخر، `float32` هو ما
تريده.

</div>


## Setup

Run the cell below first. It imports torch, loads the data and yesterday's two artefacts, and prints
the device you are on.

<div dir="rtl" align="right">

## الإعداد

شغّل الخليّة أدناه أولًا. تستورد torch وتُحمّل البيانات وأثرَي الأمس، وتطبع الجهاز الذي تعمل عليه.

</div>


In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions, device
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("torch", "matplotlib", "pyarrow")
seed_everything(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from aiep.viz import use_course_style, PALETTE
use_course_style()

moons = pd.read_parquet(get_dataset("moons_toy"))
X_np = moons[["x1", "x2"]].to_numpy()
y_np = moons["label"].to_numpy().reshape(-1, 1).astype(float)

# Yesterday's run: the weights it started from, and the loss it recorded.
saved = np.load(load_artefact("numpy_net_params.npz"))
numpy_history = pd.read_parquet(load_artefact("numpy_net_history.parquet"))

N_IN, N_HIDDEN, N_OUT = 2, 8, 1
ITERATIONS, LR = 2000, 1.0
DEVICE = device()

print(f"X {X_np.shape} | y {y_np.shape} | device: {DEVICE}")
print(f"yesterday's arrays: {sorted(saved.files)}")
print(f"yesterday's loss: {numpy_history.iloc[0]['loss']:.6f} -> "
      f"{numpy_history.iloc[-1]['loss']:.6f} over {len(numpy_history):,} iterations")
print("\n", versions())

## Section 1 — Warm-up: a tensor, and one call  (≈25 min)

Everything in this section already works.

A tensor is a NumPy array with two extra abilities: it **remembers** the operations applied to it,
and it can **move** to another device. Everything else you know about arrays transfers unchanged —
indexing, broadcasting, `.shape`, arithmetic.

The cell below makes one from yesterday's weight array and reads the two things you will read a
thousand times this bootcamp: its shape and its dtype.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: مُوتِّر ونداء واحد (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا.

المُوتِّر مصفوفة NumPy بقدرتين إضافيتين: **يتذكّر** العمليات المطبَّقة عليه، ويستطيع **الانتقال** إلى
جهاز آخر. وكل ما تعرفه عن المصفوفات ينتقل بلا تغيير — الفهرسة والبثّ و`.shape` والحساب.

وتصنع الخليّة أدناه مُوتِّرًا من مصفوفة أوزان الأمس، وتقرأ الشيئين اللذين ستقرؤهما ألف مرة في هذا
المعسكر: شكله ونوعه.

</div>


In [ ]:
W1_tensor = torch.tensor(saved["init_W1"])

print(f"from NumPy:  {saved['init_W1'].shape}, dtype {saved['init_W1'].dtype}")
print(f"as a tensor: {tuple(W1_tensor.shape)}, dtype {W1_tensor.dtype}")
print(f"it lives on: {W1_tensor.device}")
print()
print(f"torch.tensor([1.0, 2.0]).dtype is {torch.tensor([1.0, 2.0]).dtype} "
      f"— PyTorch's default, and NOT NumPy's")

# Moving to an accelerator is one call — and on a GPU it is also a dtype decision.
moved = W1_tensor.float().to(DEVICE)
print(f"\n.float().to({DEVICE!r}) -> {moved.device}, dtype {moved.dtype}")
try:
    W1_tensor.to(DEVICE)
    print(f"and float64 is fine on this device too")
except TypeError as exc:
    print(f"and float64 is NOT: {type(exc).__name__}: {str(exc).split('.')[0]}.")
print("\nGPU and Apple-silicon backends are built for float32; float64 is a CPU luxury.")
print("Today's model stays on the CPU as .double(), because the point of today is to")
print("reproduce yesterday's float64 numbers exactly. Every other week: float32.")

### Task 1.1 — one call, and Wednesday's two numbers

Yesterday you measured `∂L/∂W2[0] = −0.1360` by running the forward pass twice and dividing. Then you
wrote the analytic version and checked all 33 gradients against the numerical ones.

Here is the same network, as tensors, with `requires_grad=True` on everything you want a gradient
for. Run the forward pass — the same four operations — and then call `loss.backward()`.

Nothing is printed and nothing is returned. Look at `.grad` afterwards.

<div dir="rtl" align="right">

### المهمة ١٫١ — نداء واحد ورقما الأربعاء

قِست بالأمس `∂L/∂W2[0] = −0.1360` بتشغيل المرور الأمامي مرتين وبالقسمة. ثم كتبت النسخة التحليلية
وفحصت الاشتقاقات الثلاثة والثلاثين كلها مقابل العددية.

وهذه هي الشبكة نفسها كمُوتِّرات، مع `requires_grad=True` على كل ما تريد له اشتقاقًا. شغّل المرور
الأمامي — العمليات الأربع نفسها — ثم نادِ `loss.backward()`.

ولا يُطبع شيء ولا يُرجَع شيء. انظر في `.grad` بعدها.

</div>


In [ ]:
x = torch.tensor([1.0, 2.0], dtype=torch.float64)
W1 = torch.tensor([[0.5, -0.5], [1.0, 0.5]], dtype=torch.float64, requires_grad=True)
b1 = torch.tensor([0.0, 0.5], dtype=torch.float64, requires_grad=True)
W2 = torch.tensor([1.0, -1.0], dtype=torch.float64, requires_grad=True)
b2 = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)

a1 = torch.relu(x @ W1 + b1)
out = torch.sigmoid(a1 @ W2 + b2)
loss = (out - 1.0) ** 2

print(f"hidden layer: {a1.detach().numpy()}   <- Tuesday's [2.5, 1.0]")
print(f"output:       {out.item():.4f}          <- Tuesday's 0.8176")
print(f"loss:         {loss.item():.4f}          <- Wednesday's 0.0333")
print(f"\nthe output remembers what made it: {out.grad_fn}")

loss.backward()

print(f"\nW2.grad = {W2.grad.numpy().round(4)}")
print(f"Wednesday's hand-computed pair: [-0.1360, -0.0544]")
print(f"agree to four decimals: "
      f"{np.allclose(W2.grad.numpy().round(4), [-0.1360, -0.0544])}")
print(f"\nand every other gradient came for free:")
print(f"  W1.grad shape {tuple(W1.grad.shape)}, b1.grad {b1.grad.numpy().round(4)}, "
      f"b2.grad {b2.grad.item():.4f}")

One line, and every gradient in the graph.

Two things worth being precise about, because they are where the mysticism creeps in:

**It is not doing anything cleverer than you did.** `backward()` walks the record of operations the
forward pass built, in reverse, applying exactly the four steps you wrote yesterday. Your `dz2`,
`dW2`, `da1`, `dz1` are the same quantities, in the same order.

**The agreement is not exact by construction.** Autograd gives the analytic gradient; Wednesday's
`−0.1360` came from nudging a weight by 0.001 and dividing. They agree to four decimals because you
used a *central* difference at a small `eps` — and because, as you found yesterday, no hidden unit sat
close enough to its kink to be flipped by that nudge.

<div dir="rtl" align="right">

سطر واحد، وكل اشتقاق في الرسم البياني.

وشيئان يستحقّان الدقة، لأنهما موضع تسلّل الغموض:

**لا تفعل شيئًا أذكى مما فعلت.** فـ `backward()` تمشي في سجلّ العمليات الذي بناه المرور الأمامي
عكسيًا، مطبِّقةً الخطوات الأربع نفسها التي كتبتها بالأمس. و`dz2` و`dW2` و`da1` و`dz1` عندك هي المقادير
نفسها بالترتيب نفسه.

**والتوافق ليس تامًّا بحكم البناء.** فالاشتقاق التلقائي يعطي الاشتقاق التحليلي، أما `−0.1360` يوم
الأربعاء فجاء من تحريك وزن بمقدار ٠٫٠٠١ وقسمة. وهما يتوافقان إلى أربع خانات لأنك استخدمت فرقًا
**مركزيًا** بـ `eps` صغير — ولأن أي وحدة مخفية، كما رأيت بالأمس، لم تكن قريبة من انكسارها بما يكفي
ليقلبها ذلك التحريك.

</div>


## Section 2 — Core: six tasks  (≈60 min)

1. `TensorDataset` and `DataLoader`, and what a batch actually looks like.
2. The network as an `nn.Module`, loaded with yesterday's exact starting weights.
3. The five-line loop, full-batch, for the same 2,000 iterations.
4. Both loss curves on one figure, and the size of the disagreement.
5. Count the lines you deleted, and name where they came from.
6. Delete `optimizer.zero_grad()` on purpose and look at the damage.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `TensorDataset` و`DataLoader`، وكيف تبدو الدفعة فعلًا.
٢. الشبكة كـ `nn.Module` مُحمَّلة بأوزان الأمس الابتدائية بالضبط.
٣. حلقة الأسطر الخمسة بدفعة كاملة للتكرارات الألفين نفسها.
٤. منحنيا الخسارة في شكل واحد، وحجم الاختلاف.
٥. عُدّ الأسطر التي حذفتها وسمِّ من أين جاءت.
٦. احذف `optimizer.zero_grad()` عن قصد وانظر في الضرر.

</div>


### Task 2.1 — the conveyor belt

`TensorDataset` pairs your features with your labels and answers two questions: how many items are
there, and what is item *i*. That is the entire `Dataset` interface.

`DataLoader` wraps it and hands the training loop batches, shuffled. Set `batch_size=32` and iterate
once to see what comes out.

500 does not divide by 32. The last batch has **20** rows, and PyTorch hands it to you rather than
dropping it — which is correct, and also means any code of yours that assumes a fixed batch size is
wrong once per epoch. Print the sizes and see it.

<div dir="rtl" align="right">

### المهمة ٢٫١ — الحزام الناقل

يُزاوج `TensorDataset` خصائصك مع تصنيفاتك، ويجيب عن سؤالين: كم عدد العناصر، وما العنصر رقم *i*. وهذه
هي واجهة `Dataset` كلها.

ويلفّه `DataLoader` ويناول حلقة التدريب دفعات مخلوطة. اضبط `batch_size=32` ومُرّ مرة واحدة لترى ما
يخرج.

و٥٠٠ لا تقبل القسمة على ٣٢. فالدفعة الأخيرة فيها **٢٠** صفًا، ويناولك PyTorch إياها ولا يرميها — وهذا
صحيح، ويعني أيضًا أن أي شيفرة عندك تفترض حجم دفعة ثابتًا تكون خطأً مرة في كل حقبة. اطبع الأحجام
وانظر.

</div>


In [ ]:
X = torch.tensor(X_np, dtype=torch.float64)
y = torch.tensor(y_np, dtype=torch.float64)
BATCH_SIZE = 32

# TODO: Build a TensorDataset and a shuffled DataLoader with batch_size=32.
# مهمة: ابنِ `TensorDataset` و`DataLoader` مخلوطًا بـ `batch_size=32`.

# TODO: Iterate once and collect the shape of every batch.
# مهمة: مُرّ مرة واحدة واجمع شكل كل دفعة.

print(f"{len(dataset)} rows, batch_size {BATCH_SIZE} -> {len(batch_shapes)} batches")
print(f"first batch: {batch_shapes[0]}")
print(f"last batch:  {batch_shapes[-1]}   <- 500 = 15 x 32 + 20")
print(f"every row appears exactly once per epoch: "
      f"{sum(shape[0] for shape in batch_shapes) == len(dataset)}")

### Task 2.2 — the network as an object

Yesterday your network was a dict of arrays and two functions that knew how to use it. Ask that dict
"what are your parameters" and there is no answer — you would have to go and collect them, by name,
correctly, every time you wanted to save it, move it or optimise it.

`nn.Module` is a box that knows what is inside it. `__init__` declares the layers; `forward` says how
they compose; and because the layers were declared, `.parameters()`, `.to(device)` and
`.state_dict()` all become possible.

Then the part that makes today's comparison honest: **load yesterday's starting weights into it.**

`nn.Linear(2, 8)` stores its weight as `(out_features, in_features)` — `(8, 2)` — because it computes
`x @ W.T + b`. Yours was `(2, 8)`. So the numbers are the same numbers, transposed. Get this backwards
and nothing errors: `(2, 8)` and `(8, 2)` are both valid shapes for *something*, and you will train a
network that is quietly not yours.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — الشبكة ككائن

كانت شبكتك بالأمس قاموس مصفوفات ودالتين تعرفان كيف تستخدمانه. واسأل ذلك القاموس «ما معاملاتك» فلا
جواب — بل عليك أن تذهب وتجمعها بالاسم وبصواب في كل مرة تريد حفظها أو نقلها أو تحسينها.

و`nn.Module` صندوق يعرف ما بداخله. فـ `__init__` تُعلن الطبقات، و`forward` تقول كيف تتركّب، ولأن
الطبقات أُعلنت صار `.parameters()` و`.to(device)` و`.state_dict()` ممكنة.

ثم الجزء الذي يجعل مقارنة اليوم صادقة: **حمّل أوزان الأمس الابتدائية فيه.**

يخزّن `nn.Linear(2, 8)` وزنه بالشكل (`out_features`, `in_features`) أي `(8, 2)`، لأنه يحسب
`x @ W.T + b`. وشكلك كان `(2, 8)`. فالأرقام هي الأرقام نفسها منقولةً. وإن عكستها لم يُخرِج شيء خطأً:
فـ `(2, 8)` و`(8, 2)` شكلان صحيحان لـ**شيء ما**، وستدرّب شبكة ليست شبكتك بهدوء.

</div>


In [ ]:

# TODO: Define the 2-8-1 network as an nn.Module.
# مهمة: عرّف شبكة ٢-٨-١ كـ `nn.Module`.

model = Net().double()

# TODO: Load yesterday's starting weights into the model — mind the transpose.
# مهمة: حمّل أوزان الأمس الابتدائية في النموذج — وانتبه للنقل.

print(f"parameter shapes: {[tuple(p.shape) for p in model.parameters()]}")
n_parameters = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_parameters}   <- Tuesday's hand count for 2-8-1")
print(f"\nyour W1 was {saved['init_W1'].shape}, "
      f"nn.Linear stores {tuple(model.fc1.weight.shape)} — the same numbers, "
      f"transposed: "
      f"{np.allclose(model.fc1.weight.detach().numpy().T, saved['init_W1'])}")
print(f"\nthe untrained model on row 0: {model(X[:1]).item():.10f}")

### Task 2.3 — the five lines

```python
optimizer.zero_grad()      # clear last step's gradients
out = model(x)             # forward, building the graph as it runs
loss = criterion(out, y)   # one number
loss.backward()            # walk the graph, fill every .grad
optimizer.step()           # w <- w - lr * grad, for every parameter
```

That is the whole of training, and it is the same five statements for every model in weeks 4 to 7.

Run it full-batch — all 500 rows per step, `SGD` at `lr=1.0`, `BCELoss`, 2,000 iterations. Those are
yesterday's settings exactly, because the point of this run is not to train well, it is to train
**identically**.

Record the loss each iteration.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — الأسطر الخمسة

```python
optimizer.zero_grad()      # امسح اشتقاقات الخطوة السابقة
out = model(x)             # مرور أمامي يبني الرسم أثناء تنفيذه
loss = criterion(out, y)   # رقم واحد
loss.backward()            # امشِ في الرسم واملأ كل ‎.grad
optimizer.step()           # w <- w - lr * grad لكل معامل
```

هذا هو التدريب كله، وهي الجُمل الخمس نفسها لكل نموذج في الأسابيع من الرابع إلى السابع.

شغّلها بدفعة كاملة — الصفوف الخمسمئة كلها في كل خطوة، و`SGD` بـ `lr=1.0`، و`BCELoss`، وألفا تكرار.
وهذه إعدادات الأمس بالضبط، لأن مقصود هذا التشغيل ليس التدريب الجيد بل التدريب **المطابق**.

وسجّل الخسارة في كل تكرار.

</div>


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
criterion = nn.BCELoss()

# TODO: Write the five-line loop, full batch, for ITERATIONS steps, recording the loss.
# مهمة: اكتب حلقة الأسطر الخمسة بدفعة كاملة لعدد `ITERATIONS` خطوة مع تسجيل الخسارة.

with torch.no_grad():
    torch_accuracy = float(((model(X) > 0.5) == (y > 0.5)).double().mean())

print(f"iteration    0: {torch_losses[0]:.6f}")
print(f"iteration    1: {torch_losses[1]:.6f}")
print(f"iteration {ITERATIONS - 1}: {torch_losses[-1]:.6f}")
print(f"final accuracy: {torch_accuracy:.3f}")

### Task 2.4 — the two curves

Now put yesterday's numbers next to today's.

Plot both loss curves on one figure, then print the first three of each, side by side, and the
largest disagreement across all 2,000 iterations.

Slide 59 claimed they would be identical to six decimals. Check that claim yourself, and report what
you actually find rather than what you were told to expect.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — المنحنيان

وضَع الآن أرقام الأمس بجانب أرقام اليوم.

ارسم منحنيَي الخسارة في شكل واحد، ثم اطبع أول ثلاثة من كلٍّ منهما جنبًا إلى جنب، وأكبر اختلاف عبر
التكرارات الألفين كلها.

ادّعت الشريحة ٥٩ أنهما سيكونان مطابقين إلى ست خانات عشرية. تحقّق من الادعاء بنفسك، واعرض ما وجدته
فعلًا لا ما قيل لك أن تتوقّعه.

</div>


In [ ]:
numpy_losses = numpy_history["loss"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TODO: Plot both curves on the left panel and their absolute difference on the right.
# مهمة: ارسم المنحنيين في اللوحة اليسرى والفرق المطلق بينهما في اليمنى.

plt.tight_layout()
plt.show()

max_gap = float(np.abs(numpy_losses - torch_losses).max())
print(f"{'iteration':<10}{'NumPy':>14}{'PyTorch':>14}")
for i in (0, 1, 2):
    print(f"{i:<10}{numpy_losses[i]:>14.10f}{torch_losses[i]:>14.10f}")
print(f"\nlargest disagreement over {ITERATIONS:,} iterations: {max_gap:.2e}")
print(f"identical to six decimals: {max_gap < 5e-7}")

The two runs agree to about **1e-15**, which is the precision of `float64` itself. They are not
similar; they are the same computation, expressed twice.

That is worth being clear about, because it is the whole claim of the day. PyTorch did not use a
better algorithm, a cleverer gradient, or a smarter update. It did exactly what you wrote on
Wednesday, on the same starting numbers, in the same order — and the only reason the agreement is
this exact is that you loaded your own initial weights instead of letting it initialise its own.

Had you skipped that step, the two curves would have differed visibly from iteration 1, and you would
have spent the afternoon debugging a difference that was only initialisation. Which is the pitfall on
slide 62, and the reason `numpy_net_params.npz` exists.

<div dir="rtl" align="right">

يتوافق التشغيلان إلى نحو **1e-15**، وهي دقة `float64` نفسها. فهما ليسا متشابهين، بل هما الحساب نفسه
مكتوبًا مرتين.

وهذا يستحقّ الوضوح، فهو ادعاء اليوم كله. فلم يستخدم PyTorch خوارزمية أفضل ولا اشتقاقًا أذكى ولا
تحديثًا أحكم. بل فعل بالضبط ما كتبته يوم الأربعاء، على الأرقام الابتدائية نفسها وبالترتيب نفسه — والسبب
الوحيد لهذه الدقة في التوافق أنك حمّلت أوزانك الابتدائية بدل أن تدعه يُهيّئ أوزانه.

ولو تخطّيت تلك الخطوة لاختلف المنحنيان اختلافًا مرئيًا من التكرار الأول، ولصرفت بعد الظهر في تصحيح
فرقٍ لم يكن إلا التهيئة. وهذا هو الفخّ في الشريحة ٦٢، وهو سبب وجود `numpy_net_params.npz`.

</div>


### Task 2.5 — count what went

Yesterday's `backward()` was about 20 lines. Today's replacement is `loss.backward()`.

Count them honestly. Take yesterday's backward pass, count its non-blank lines, and compare with the
five-line loop. Then be precise about what actually disappeared: `nn.Linear` **renamed** your matmul,
`torch.relu` **renamed** `np.maximum(0, z)`, `nn.BCELoss` **renamed** your formula, `optimizer.step()`
**renamed** your update. Only one row of that table is a real removal.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — عُدّ ما ذهب

كانت `backward()` بالأمس نحو عشرين سطرًا. وبديلها اليوم `loss.backward()`.

عُدّها بصدق. خُذ مرور الأمس الخلفي وعُدّ أسطره غير الفارغة وقارنها بحلقة الأسطر الخمسة. ثم كُن دقيقًا
في ما اختفى فعلًا: فـ `nn.Linear` **أعادت تسمية** ضربك المصفوفي، و`torch.relu` **أعادت تسمية**
`np.maximum(0, z)`، و`nn.BCELoss` **أعادت تسمية** صيغتك، و`optimizer.step()` **أعادت تسمية** تحديثك.
وصفّ واحد فقط من ذلك الجدول حذفٌ حقيقي.

</div>


In [ ]:
yesterday_backward = """
def backward(params, cache, X, y, output):
    n = len(X)
    dz2 = (output - y) / n
    dW2 = cache["a1"].T @ dz2
    db2 = dz2.sum(axis=0)
    da1 = dz2 @ params["W2"].T
    dz1 = da1 * (cache["z1"] > 0)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}
"""

today_loop = """
optimizer.zero_grad()
output = model(X)
loss = criterion(output, y)
loss.backward()
optimizer.step()
"""

backward_lines = len([line for line in yesterday_backward.strip().splitlines() if line.strip()])
loop_lines = len([line for line in today_loop.strip().splitlines() if line.strip()])

print(f"your backward pass:      {backward_lines} lines")
print(f"the PyTorch loop:        {loop_lines} lines")
print(f"and one of those five is the replacement: loss.backward()")
print()
print(f"{'your NumPy':<34}{'PyTorch':<24}{'what was taken'}")
print("-" * 76)
for numpy_side, torch_side, taken in [
    ("z1 = X @ W1 + b1", "nn.Linear(2, 8)", "the matmul and the init"),
    ("a1 = np.maximum(0, z1)", "torch.relu", "nothing — renamed"),
    ("-mean(y*log(p) + ...)", "nn.BCELoss()", "nothing — renamed"),
    (f"{backward_lines} lines of backward pass", "loss.backward()", "ALL of it"),
    ("params[k] -= LR * grads[k]", "optimizer.step()", "the bookkeeping"),
]:
    print(f"{numpy_side:<34}{torch_side:<24}{taken}")

### Task 2.6 — break it on purpose

PyTorch **accumulates** gradients into `.grad` rather than replacing them. That is deliberate: it is
what lets you split one large batch across several backward passes. It also means that if you never
clear them, step 2 uses two gradients added together, step 10 uses ten, and the effective step size
grows every iteration.

Remove `optimizer.zero_grad()`, train a fresh copy of the model for 40 iterations, and plot the loss
beside a correct 40-iteration run.

Then put the line back. This is the most common PyTorch bug there is, it raises no error, and the
curve it produces — down for a while, then away — is now something you can recognise on sight.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — اكسِرها عن قصد

يُراكم PyTorch الاشتقاقات في `.grad` ولا يستبدلها. وهذا مقصود: فهو ما يتيح لك تقسيم دفعة كبيرة على
عدة مرورات خلفية. ويعني أيضًا أنك إن لم تمسحها استخدمت الخطوة الثانية اشتقاقين مجموعين، والخطوة
العاشرة عشرة، فيكبر حجم الخطوة الفعلي في كل تكرار.

احذف `optimizer.zero_grad()`، ودرّب نسخة جديدة من النموذج أربعين تكرارًا، وارسم الخسارة بجانب تشغيل
صحيح من أربعين تكرارًا.

ثم أعِد السطر. فهذا أشيع عيب في PyTorch، ولا يُخرِج خطأً، والمنحنى الذي يُنتجه — هبوط ثم ابتعاد — صار
شيئًا تعرفه من النظرة الأولى.

</div>


In [ ]:
BREAK_LR = 2 * LR   # see the note below the plot


def fresh_model():
    """A new model loaded with yesterday's starting weights. Both runs start here."""
    m = Net().double()
    with torch.no_grad():
        m.fc1.weight.copy_(torch.tensor(saved["init_W1"].T))
        m.fc1.bias.copy_(torch.tensor(saved["init_b1"]))
        m.fc2.weight.copy_(torch.tensor(saved["init_W2"].T))
        m.fc2.bias.copy_(torch.tensor(saved["init_b2"]))
    return m


# TODO: Write train_briefly(iterations, zero_grad=True, lr=BREAK_LR) -> losses.
# مهمة: اكتب `train_briefly(iterations, zero_grad=True, lr=BREAK_LR)` ← الخسائر.

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(correct_run, color=PALETTE[0], linewidth=2, label="with zero_grad()")
ax.semilogy(broken_run, color=PALETTE[3], linewidth=2,
            label="without zero_grad()")
ax.set_xlabel("iteration")
ax.set_ylabel("loss (log scale)")
ax.set_title("One missing line")
ax.legend()
plt.show()

print(f"with zero_grad:    {correct_run[0]:.4f} -> {correct_run[-1]:.4f}")
print(f"without zero_grad: {broken_run[0]:.4f} -> {broken_run[-1]:.4f}")
print(f"the broken run's final loss is "
      f"{broken_run[-1] / correct_run[-1]:.1f}x the correct one's")

Down, and then away. The broken run peaks around 2.8 — four times its starting loss — and ends four
times worse than the correct one, from removing a single line that raised no error.

Two notes on how this was set up, because both matter more than the picture.

**The learning rate here is twice the one you have been using.** At `lr=1.0` on a problem this small,
the accumulated gradient behaves mostly like a bigger step and the run sometimes ends *better* than
the correct one for a while — which is exactly what makes this bug dangerous. It does not always look
like a disaster; it looks like a run that trained oddly. Doubling the rate makes the mechanism visible
inside 40 iterations instead of 400.

**The effective step size grows every iteration.** Step 1 uses one gradient, step 2 uses two added
together, step 10 uses ten. So the early iterations look normal and the damage compounds — which is
why the curve falls first and diverges later, rather than being wrong from the start.

<div dir="rtl" align="right">

هبوط ثم ابتعاد. فالتشغيل المعطوب يبلغ ذروته عند نحو ٢٫٨ — أي أربعة أضعاف خسارته الابتدائية — وينتهي
أسوأ من الصحيح بأربعة أضعاف، بحذف سطر واحد لم يُخرِج خطأً.

وملاحظتان على طريقة الإعداد، وكلتاهما أهمّ من الصورة.

**معدّل التعلّم هنا ضعف الذي كنت تستخدمه.** فعند `lr=1.0` على مسألة بهذا الصغر يتصرّف الاشتقاق
المتراكم كخطوة أكبر في الأغلب، وينتهي التشغيل أحيانًا **أفضل** من الصحيح لفترة — وهذا بالضبط ما يجعل
هذا العيب خطِرًا. فهو لا يبدو كارثة دائمًا، بل يبدو تشغيلًا تدرّب على نحو غريب. ومضاعفة المعدّل تُظهر
الآلية في أربعين تكرارًا بدل أربعمئة.

**وحجم الخطوة الفعلي يكبر في كل تكرار.** فالخطوة الأولى تستخدم اشتقاقًا واحدًا، والثانية اثنين
مجموعين، والعاشرة عشرة. فتبدو التكرارات الأولى عادية ويتراكم الضرر — ولهذا يهبط المنحنى أولًا ثم
يبتعد، بدل أن يكون خطأً من البداية.

</div>


**Your sentence:** _(what accumulated, and why does the damage grow with every iteration rather than
staying constant?)_

<div dir="rtl" align="right">

**جملتك:** _(ما الذي تراكم، ولماذا يكبر الضرر مع كل تكرار بدل أن يبقى ثابتًا؟)_

</div>


## Section 3 — Stretch: mini-batches, and the fused loss  (≈30 min)

Open-ended. Lower expectation of completeness.

Two things, and the first one produces today's artefact.

**Mini-batches.** Train the same network through the `DataLoader` from task 2.1 — 32 rows per step,
shuffled, on a proper train/validation split — and record both losses per epoch. The curve is noisier
than the full-batch one and it gets there in far fewer passes over the data, which is Sunday's whole
first section arriving early.

**`BCEWithLogitsLoss`.** Remove the sigmoid from the model's output and use
`nn.BCEWithLogitsLoss` instead of `nn.BCELoss`. Mathematically it is the same loss; numerically it is
safer, because it computes the sigmoid and the logarithm together and never forms a probability that
could round to exactly 0 or 1. This is why every serious PyTorch codebase uses it, and why models you
read will often have no activation on their final layer.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: الدفعات المصغّرة والخسارة المدموجة (نحو ٣٠ دقيقة)

مفتوح. والتوقّع في الإكمال أقل.

أمران، والأول يُنتج أثر اليوم.

**الدفعات المصغّرة.** درّب الشبكة نفسها عبر `DataLoader` من المهمة ٢٫١ — ٣٢ صفًا في الخطوة، مخلوطة،
على تقسيم تدريب وتحقّق حقيقي — وسجّل الخسارتين في كل حقبة. والمنحنى أكثر ضوضاءً من منحنى الدفعة
الكاملة ويصل بعدد مرورات أقل بكثير على البيانات، وهذا هو القسم الأول من يوم الأحد كله وقد وصل مبكرًا.

**`BCEWithLogitsLoss`.** احذف الدالة السينية من مَخرج النموذج واستخدم `nn.BCEWithLogitsLoss` بدلًا من
`nn.BCELoss`. وهي رياضيًا الخسارة نفسها، وعدديًا أأمن، لأنها تحسب الدالة السينية واللوغاريتم معًا ولا
تُكوّن احتمالًا قد يُقرَّب إلى صفر أو واحد تامّ. ولهذا تستخدمها كل شيفرة PyTorch جادّة، ولهذا كثير من
النماذج التي تقرؤها لا تحمل دالة تنشيط على طبقتها الأخيرة.

</div>


In [ ]:
EPOCHS = 200
torch.manual_seed(42)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(42))
train_idx, val_idx = perm[:400], perm[400:]
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

# TODO: Build a shuffled training loader and train for EPOCHS, recording both losses.
# مهمة: ابنِ مُحمّل تدريب مخلوطًا ودرّب `EPOCHS` حقبة مع تسجيل الخسارتين.

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(torch_history["epoch"], torch_history["train_loss"],
        color=PALETTE[0], label="training")
ax.plot(torch_history["epoch"], torch_history["val_loss"],
        color=PALETTE[1], label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")
ax.set_title(f"Mini-batch training, {BATCH_SIZE} rows per step")
ax.legend()
plt.show()

print(f"after {EPOCHS} epochs ({EPOCHS * len(train_loader):,} steps): "
      f"train {torch_history.iloc[-1]['train_loss']:.4f}, "
      f"validation {torch_history.iloc[-1]['val_loss']:.4f}, "
      f"val accuracy {torch_history.iloc[-1]['val_accuracy']:.3f}")

In [ ]:

# TODO: sigmoid + BCELoss model after 300 identical steps.
# مهمة: `BCELoss` بعد ٣٠٠ خطوة متطابقة.

print(f"largest difference between the fused and the plain model: "
      f"{fused_vs_plain:.2e}")
print(f"the same model, and the fused one never forms a probability at all")
print(f"\nmodel.eval() changed the predictions: {not eval_unchanged}")
print(f"— no dropout and no batch norm yet, so it cannot. On Sunday it will.")

**Your two sentences:** _(why is computing the sigmoid and the logarithm together safer than computing
them one after the other? And what would you have to add to the model before `eval()` starts
mattering?)_

<div dir="rtl" align="right">

**جملتاك:** _(لماذا يكون حساب الدالة السينية واللوغاريتم معًا أأمن من حسابهما واحدًا بعد الآخر؟ وما
الذي عليك إضافته إلى النموذج قبل أن يبدأ `eval()` بالتأثير؟)_

</div>


## Save your artefact

`torch_net_history.parquet` — one row per epoch of the mini-batch run: the epoch number, the training
loss, the validation loss and the validation accuracy.

Note what changed about the shape of this file compared with yesterday's. Yesterday you recorded a
single loss per iteration, because there was only one thing to record. Today there are **two** curves,
because there is a held-out set — and tomorrow's entire lab is about reading the gap between them.

<div dir="rtl" align="right">

## احفظ مخرجاتك

`torch_net_history.parquet` وفيه صف لكل حقبة من تشغيل الدفعات المصغّرة: رقم الحقبة، وخسارة التدريب،
وخسارة التحقّق، ودقة التحقّق.

ولاحظ ما تغيّر في شكل هذا الملف مقارنةً بملف الأمس. فقد سجّلت بالأمس خسارة واحدة لكل تكرار لأنه لم
يكن ثمّة إلا شيء واحد يُسجَّل. واليوم منحنيان لأن ثمّة مجموعة محجوزة — ومعمل الغد كله عن قراءة الفجوة
بينهما.

</div>


In [ ]:
out = ARTEFACT_DIR / "torch_net_history.parquet"
torch_history.to_parquet(out, index=False)

reloaded = pd.read_parquet(out)
print(f"Saved {out}")
print(f"{len(reloaded)} epochs, columns {list(reloaded.columns)}")
print(f"train loss {reloaded.iloc[0]['train_loss']:.4f} -> "
      f"{reloaded.iloc[-1]['train_loss']:.4f}")
print(f"val   loss {reloaded.iloc[0]['val_loss']:.4f} -> "
      f"{reloaded.iloc[-1]['val_loss']:.4f}")
best_epoch = int(reloaded['val_loss'].idxmin())
print(f"\nbest validation loss was {reloaded['val_loss'].min():.4f} at epoch {best_epoch}, "
      f"and the last epoch scored {reloaded.iloc[-1]['val_loss']:.4f}")
print(f"keep that pair in mind — it is tomorrow's early-stopping task in one line")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

The first one is the day's claim in a single assertion: a library you never taught anything printed
the gradient you computed by hand on Wednesday.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخليّة أخيرًا. كل فحص يفشل يخبرك بما تُصلحه ولماذا.

والفحص الأول هو ادعاء اليوم في تحقّق واحد: مكتبة لم تُعلّمها شيئًا طبعت الاشتقاق الذي حسبته بيدك يوم
الأربعاء.

</div>


In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(np.allclose(W2.grad.numpy().round(4), [-0.1360, -0.0544]),
      f"loss.backward() must reproduce Wednesday's hand-computed gradients "
      f"[-0.1360, -0.0544] to four decimals — it produced {W2.grad.numpy().round(6)}",
      f"يجب أن يُعيد `loss.backward()` إنتاج اشتقاقَي الأربعاء المحسوبين يدويًا "
      f"`[-0.1360, -0.0544]` إلى أربع خانات عشرية — والناتج {W2.grad.numpy().round(6)}")

check(batch_shapes[0] == (32, 2) and batch_shapes[-1] == (20, 2),
      f"a full DataLoader batch must be (32, 2) and the last one (20, 2) — got "
      f"{batch_shapes[0]} and {batch_shapes[-1]}. 500 rows do not divide into 32.",
      f"يجب أن تكون الدفعة الكاملة `(32, 2)` والأخيرة `(20, 2)` — والناتج "
      f"{batch_shapes[0]} و{batch_shapes[-1]}. فـ ٥٠٠ صف لا تقبل القسمة على ٣٢.")

check(n_parameters == 33,
      f"the model must hold 33 parameters — 24 weights and 9 biases, the count you made "
      f"by hand on Tuesday — and it holds {n_parameters}",
      f"يجب أن يحمل النموذج ٣٣ معاملًا — ٢٤ وزنًا و٩ انحيازات، وهو العدد الذي عددته بيدك "
      f"يوم الثلاثاء — والموجود {n_parameters}")

check(max_gap < 1e-6,
      f"with the same starting weights and the same settings, PyTorch must reproduce your "
      f"NumPy loss curve to six decimals — the largest disagreement over {ITERATIONS:,} "
      f"iterations was {max_gap:.2e}",
      f"بالأوزان الابتدائية نفسها والإعدادات نفسها، يجب أن يُعيد PyTorch إنتاج منحنى خسارتك "
      f"بـ NumPy إلى ست خانات عشرية — وأكبر اختلاف عبر {ITERATIONS:,} تكرار كان {max_gap:.2e}")

numpy_accuracy = float(numpy_history.iloc[-1]["accuracy"])
check(abs(torch_accuracy - numpy_accuracy) < 0.03,
      f"the PyTorch network's final accuracy must be within 3 points of the NumPy one — "
      f"{torch_accuracy:.3f} vs {numpy_accuracy:.3f}",
      f"يجب أن تكون الدقة النهائية لشبكة PyTorch في حدود ٣ نقاط من دقة شبكة NumPy — "
      f"{torch_accuracy:.3f} مقابل {numpy_accuracy:.3f}")

check(broken_run[-1] > correct_run[-1],
      f"the run without zero_grad() must end WORSE than the correct one at lr={BREAK_LR} "
      f"— broken "
      f"{broken_run[-1]:.4f} vs correct {correct_run[-1]:.4f}. If it did not, check that "
      f"you actually skipped the call rather than moving it.",
      f"يجب أن ينتهي التشغيل بلا `zero_grad()` **أسوأ** من الصحيح — المعطوب "
      f"{broken_run[-1]:.4f} مقابل الصحيح {correct_run[-1]:.4f}. فإن لم يكن كذلك فتحقّق "
      f"أنك حذفت النداء فعلًا ولم تنقله فقط.")

check(out.exists() and len(reloaded) == EPOCHS
      and {"train_loss", "val_loss"} <= set(reloaded.columns),
      f"torch_net_history.parquet must record {EPOCHS} epochs with both losses — it has "
      f"{len(reloaded)} rows and the columns {list(reloaded.columns)}",
      f"يجب أن يسجّل `torch_net_history.parquet` عدد {EPOCHS} حقبة بالخسارتين — وفيه "
      f"{len(reloaded)} صفًا والأعمدة {list(reloaded.columns)}")

report()

## What's next

You have written the same network twice and proved the two are one computation. From here you use the
library, and you use it knowing exactly what it replaced.

Tomorrow (**W3D5**) is the week's reckoning, in two halves. First, diagnosis: four loss curves, four
different faults, and the vocabulary for naming them — a learning rate too high, a learning rate too
low, overfitting, and a model that learned nothing at all and says so with the number 0.6931. Then
optimisers, dropout, batch norm and early stopping, measured rather than described.

And then the comparison this week has been building towards: your best network against Monday's
boosted tree, on `credit_default`, with both spreads. On twenty tabular columns the tree usually
wins. **If it does, you report that** — that is the honest result, and it is the marked part.

Two things travel with you:

- **The five lines.** Every model in weeks 4 to 7 is trained by them, unchanged.
- **`torch_net_history.parquet`** — the first file you have with a training curve *and* a validation
  curve. Tomorrow is about the gap between them.

**A3 is due W4D2.** Today's loop is what you will train it with.

<div dir="rtl" align="right">

## ماذا بعد

كتبت الشبكة نفسها مرتين وأثبتّ أن الاثنتين حساب واحد. ومن هنا تستخدم المكتبة، وتستخدمها وأنت تعرف
بالضبط ما الذي حلّت محلّه.

وغدًا (**الأسبوع ٣ اليوم ٥**) هو حساب الأسبوع في نصفين. أولًا التشخيص: أربعة منحنيات خسارة وأربعة
أعطال مختلفة، والمفردات التي تسمّيها بها — معدّل تعلّم مرتفع، ومعدّل منخفض، وفرط مطابقة، ونموذج لم
يتعلّم شيئًا ويقول ذلك بالرقم ٠٫٦٩٣١. ثم المُحسِّنات وDropout وتطبيع الدفعات والإيقاف المبكر، مقيسةً
لا موصوفة.

ثم المقارنة التي يبنيها هذا الأسبوع: أفضل شبكة عندك مقابل شجرة الاثنين المعزَّزة، على
`credit_default`، بمدى تفاوت كلٍّ منهما. وعلى عشرين عمودًا جدوليًا تفوز الشجرة عادةً. **فإن فازت
فأبلِغ بذلك** — فهذه هي النتيجة الصادقة وهي الجزء الذي يُقيَّم.

وشيئان يسافران معك:

- **الأسطر الخمسة.** فكل نموذج في الأسابيع من الرابع إلى السابع يُدرَّب بها بلا تغيير.
- **`torch_net_history.parquet`** وهو أول ملف عندك فيه منحنى تدريب **ومنحنى تحقّق**. والغد عن الفجوة
  بينهما.

**ويُسلَّم التكليف الثالث في الأسبوع الرابع اليوم الثاني.** وحلقة اليوم هي ما تدرّبه بها.

</div>
